# NeuroGolf submission builder

`outputs/` と `Sample/submission/` の両方に存在する `taskXXX.onnx` だけを比較し、推定scoreが良い方を `Sample/submission/` 側に反映して、最終 `submission.zip` を作成する notebook。

In [1]:
from pathlib import Path
import copy
import importlib.util
import json
import math
import os
import re
import shutil
import time
import traceback
import zipfile

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "neurogolf-2026").is_dir() and (path / "outputs").is_dir():
            return path
    raise FileNotFoundError("project root not found")


ROOT = find_project_root()
DATA_DIR = ROOT / "neurogolf-2026"
OUTPUT_DIR = ROOT / "outputs"
SAMPLE_SUBMISSION_DIR = ROOT / "Sample" / "submission"
PROFILE_DIR = ROOT / "profiles" / "submission-builder"
BACKUP_DIR = ROOT / "artifacts" / "sample-submission-backups" / time.strftime("%Y%m%d_%H%M%S")
FINAL_ZIP = OUTPUT_DIR / "submission.zip"
MPL_CACHE_DIR = ROOT / ".matplotlib-cache"

PROFILE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
MPL_CACHE_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

spec = importlib.util.spec_from_file_location(
    "neurogolf_utils",
    DATA_DIR / "neurogolf_utils" / "neurogolf_utils.py",
)
ng = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ng)
ng._NEUROGOLF_DIR = str(DATA_DIR) + "/"

FILESIZE_LIMIT = int(1.44 * 1024 * 1024)
print("ROOT", ROOT)
print("OUTPUT_DIR", OUTPUT_DIR)
print("SAMPLE_SUBMISSION_DIR", SAMPLE_SUBMISSION_DIR)
print("FINAL_ZIP", FINAL_ZIP)

ROOT /Users/kaiikeda/Programming/Kaggle/Neuro Golf
OUTPUT_DIR /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs
SAMPLE_SUBMISSION_DIR /Users/kaiikeda/Programming/Kaggle/Neuro Golf/Sample/submission
FINAL_ZIP /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/submission.zip


In [2]:
def task_num_from_path(path: Path) -> int:
    m = re.search(r"task(\d{3})\.onnx$", path.name)
    if not m:
        raise ValueError(f"unexpected ONNX filename: {path}")
    return int(m.group(1))


def load_examples(task_num: int):
    with open(DATA_DIR / f"task{task_num:03d}.json") as f:
        return json.load(f)


def verify_subset(session, examples):
    passed = 0
    failed = 0
    skipped = 0
    first_failure = None
    for idx, example in enumerate(examples):
        benchmark = ng.convert_to_numpy(example)
        if benchmark is None:
            skipped += 1
            continue
        try:
            actual = ng.run_network(session, benchmark["input"])
            if np.array_equal(actual, benchmark["output"]):
                passed += 1
            else:
                failed += 1
                if first_failure is None:
                    first_failure = {"idx": idx, "example": example, "actual": ng.convert_from_numpy(actual)}
        except Exception as exc:
            failed += 1
            if first_failure is None:
                first_failure = {"idx": idx, "error": repr(exc), "traceback": traceback.format_exc()}
    return passed, failed, skipped, first_failure


def evaluate_onnx(path: Path, source: str):
    started = time.time()
    task_num = task_num_from_path(path)
    filesize = path.stat().st_size
    examples = load_examples(task_num)
    model = onnx.load(path)
    onnx.checker.check_model(model, full_check=True)
    sanitized = ng.sanitize_model(copy.deepcopy(model))
    if sanitized is None:
        raise ValueError("sanitize_model failed")

    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    options.profile_file_prefix = str(PROFILE_DIR / f"{source}_task{task_num:03d}")
    session = ort.InferenceSession(sanitized.SerializeToString(), options)

    agi_pass, agi_fail, agi_skip, agi_failure = verify_subset(session, examples["train"] + examples["test"])
    gen_pass, gen_fail, gen_skip, gen_failure = verify_subset(session, examples["arc-gen"])
    profile_path = session.end_profiling()
    memory, params = ng.score_network(sanitized, profile_path)
    score = None
    if memory is not None and params is not None and memory >= 0 and params >= 0:
        score = max(1.0, 25.0 - math.log(max(1.0, memory + params)))

    return {
        "task": task_num,
        "source": source,
        "path": path,
        "onnx": str(path.relative_to(ROOT)),
        "filesize": filesize,
        "size_ok": filesize <= FILESIZE_LIMIT,
        "arc_agi_pass": agi_pass,
        "arc_agi_fail": agi_fail,
        "arc_agi_skip": agi_skip,
        "arc_gen_pass": gen_pass,
        "arc_gen_fail": gen_fail,
        "arc_gen_skip": gen_skip,
        "ready": filesize <= FILESIZE_LIMIT and agi_fail == 0 and gen_fail == 0,
        "memory_bytes": memory,
        "params": params,
        "estimated_score": score,
        "profile_path": profile_path,
        "seconds": time.time() - started,
        "first_failure": agi_failure or gen_failure,
    }

In [3]:
output_files = {task_num_from_path(p): p for p in sorted(OUTPUT_DIR.glob("task*.onnx"))}
sample_files = {task_num_from_path(p): p for p in sorted(SAMPLE_SUBMISSION_DIR.glob("task*.onnx"))}
common_tasks = sorted(set(output_files) & set(sample_files))

print(f"outputs: {len(output_files)}")
print(f"sample: {len(sample_files)}")
print(f"common: {len(common_tasks)}")
common_tasks[:30], common_tasks[-10:]

outputs: 11
sample: 400
common: 11


([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 255], [2, 3, 4, 5, 6, 7, 8, 9, 10, 255])

## Compare only common tasks

`outputs/` と `Sample/submission/` の両方に同じ `taskXXX.onnx` があるtaskだけを評価する。task8/task9/task255のような重いモデルが含まれる場合は時間がかかる。

In [4]:
def evaluate_common_tasks(tasks=common_tasks):
    rows = []
    failures = {}
    for task_num in tasks:
        for source, path in [("outputs", output_files[task_num]), ("sample", sample_files[task_num])]:
            print(f"task {task_num:03d} {source}: validating {path.name} ...", flush=True)
            try:
                row = evaluate_onnx(path, source)
                rows.append({k: v for k, v in row.items() if k not in {"path", "first_failure"}})
                if row["first_failure"] is not None:
                    failures[(task_num, source)] = row["first_failure"]
                score_text = "nan" if row["estimated_score"] is None else f"{row['estimated_score']:.3f}"
                print(
                    f"  ready={row['ready']} agi={row['arc_agi_pass']}/{row['arc_agi_pass'] + row['arc_agi_fail']} "
                    f"gen={row['arc_gen_pass']}/{row['arc_gen_pass'] + row['arc_gen_fail']} "
                    f"score={score_text} size={row['filesize']} bytes time={row['seconds']:.1f}s",
                    flush=True,
                )
            except Exception as exc:
                rows.append({
                    "task": task_num,
                    "source": source,
                    "onnx": str(path.relative_to(ROOT)),
                    "ready": False,
                    "error": repr(exc),
                })
                failures[(task_num, source)] = {"error": repr(exc), "traceback": traceback.format_exc()}
                print(f"  ERROR: {exc}", flush=True)
    return pd.DataFrame(rows).sort_values(["task", "source"]).reset_index(drop=True), failures


comparison, failures = evaluate_common_tasks()
comparison

task 001 outputs: validating task001.onnx ...
  ready=True agi=6/6 gen=262/262 score=15.272 size=3374 bytes time=0.6s
task 001 sample: validating task001.onnx ...
  ready=True agi=6/6 gen=262/262 score=17.047 size=1082 bytes time=0.4s
task 002 outputs: validating task002.onnx ...
  ready=True agi=6/6 gen=262/262 score=11.553 size=13751 bytes time=6.4s
task 002 sample: validating task002.onnx ...
  ready=True agi=6/6 gen=262/262 score=13.515 size=14560 bytes time=3.0s
task 003 outputs: validating task003.onnx ...
  ready=True agi=4/4 gen=261/261 score=13.771 size=10277 bytes time=2.6s
task 003 sample: validating task003.onnx ...
  ready=True agi=4/4 gen=261/261 score=17.918 size=5000 bytes time=1.4s
task 004 outputs: validating task004.onnx ...


2026-06-10 19:45:01.682 python[77329:11086239] 2026-06-10 19:45:01.681501 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_3'. It is not used by any node and should be removed from the model.


  ready=True agi=3/3 gen=262/262 score=12.113 size=31356 bytes time=5.8s
task 004 sample: validating task004.onnx ...
  ready=True agi=3/3 gen=262/262 score=14.085 size=2522 bytes time=0.4s
task 005 outputs: validating task005.onnx ...


2026-06-10 19:45:08.024 python[77329:11086239] 2026-06-10 19:45:08.023995 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_3'. It is not used by any node and should be removed from the model.


  ready=True agi=4/4 gen=262/262 score=10.466 size=85748 bytes time=17.2s
task 005 sample: validating task005.onnx ...
  ready=True agi=4/4 gen=262/262 score=13.328 size=10910 bytes time=3.1s
task 006 outputs: validating task006.onnx ...
  ready=True agi=4/4 gen=262/262 score=14.497 size=1540 bytes time=0.3s
task 006 sample: validating task006.onnx ...
  ready=True agi=4/4 gen=262/262 score=17.603 size=1124 bytes time=0.4s
task 007 outputs: validating task007.onnx ...
  ready=True agi=4/4 gen=262/262 score=12.356 size=21339 bytes time=2.9s
task 007 sample: validating task007.onnx ...
  ready=True agi=4/4 gen=262/262 score=16.108 size=1258 bytes time=0.4s
task 008 outputs: validating task008.onnx ...
  ready=True agi=4/4 gen=262/262 score=10.496 size=113319 bytes time=20.7s
task 008 sample: validating task008.onnx ...
  ready=True agi=4/4 gen=262/262 score=14.944 size=7647 bytes time=1.3s
task 009 outputs: validating task009.onnx ...


2026-06-10 19:45:55.557 python[77329:11086239] 2026-06-10 19:45:55.558518 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_2'. It is not used by any node and should be removed from the model.


  ready=True agi=4/4 gen=261/261 score=11.694 size=17783 bytes time=3.2s
task 009 sample: validating task009.onnx ...
  ready=True agi=4/4 gen=261/261 score=13.695 size=30956 bytes time=1.0s
task 010 outputs: validating task010.onnx ...
  ready=True agi=3/3 gen=262/262 score=12.910 size=28504 bytes time=3.8s
task 010 sample: validating task010.onnx ...
  ready=True agi=3/3 gen=262/262 score=16.480 size=6176 bytes time=1.6s
task 255 outputs: validating task255.onnx ...
  ready=True agi=4/4 gen=261/261 score=10.724 size=252290 bytes time=5.1s
task 255 sample: validating task255.onnx ...
  ready=True agi=4/4 gen=261/261 score=10.675 size=252958 bytes time=5.0s


,task,source,onnx,filesize,size_ok,arc_agi_pass,arc_agi_fail,arc_agi_skip,arc_gen_pass,arc_gen_fail,arc_gen_skip,ready,memory_bytes,params,estimated_score,profile_path,seconds
0,1,outputs,outputs/task001.onnx,3374,True,6,0,0,262,0,0,True,16620,156,15.272295,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,0.637282
1,1,sample,Sample/submission/task001.onnx,1082,True,6,0,0,262,0,0,True,2819,24,17.047385,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,0.365184
2,2,outputs,outputs/task002.onnx,13751,True,6,0,0,262,0,0,True,691404,46,11.553454,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,6.370874
3,2,sample,Sample/submission/task002.onnx,14560,True,6,0,0,262,0,0,True,97200,93,13.514518,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,3.021851
4,3,outputs,outputs/task003.onnx,10277,True,4,0,0,261,0,0,True,74308,985,13.770858,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,2.616678
5,3,sample,Sample/submission/task003.onnx,5000,True,4,0,0,261,0,0,True,1172,18,17.918291,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,1.351547
6,4,outputs,outputs/task004.onnx,31356,True,3,0,0,262,0,0,True,394040,1256,12.112610,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,5.758846
7,4,sample,Sample/submission/task004.onnx,2522,True,3,0,0,262,0,0,True,54000,1020,14.084548,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,0.369663
8,5,outputs,outputs/task005.onnx,85748,True,4,0,0,262,0,0,True,2049212,1544,10.466281,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,17.223287
9,5,sample,Sample/submission/task005.onnx,10910,True,4,0,0,262,0,0,True,116889,336,13.328150,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,3.090324


In [5]:
def choose_best(group):
    usable = group[group["ready"].fillna(False)].copy()
    if usable.empty:
        return group.assign(selected=False, selection_reason="no ready model")
    usable["score_key"] = usable["estimated_score"].fillna(-1.0)
    usable["filesize_key"] = -usable["filesize"].fillna(10**18)
    best_idx = usable.sort_values(["score_key", "filesize_key"], ascending=False).index[0]
    result = group.copy()
    result["selected"] = result.index == best_idx
    result["selection_reason"] = "best estimated_score among ready models"
    return result


selected = comparison.groupby("task", group_keys=False).apply(choose_best).reset_index(drop=True)
selected_summary = selected[selected["selected"]].copy().sort_values("task")
selected_summary[["task", "source", "estimated_score", "memory_bytes", "params", "filesize", "ready", "onnx"]]

/var/folders/06/fdzcb6v92lv3bc1kz4b4p_s00000gn/T/ipykernel_77329/1059936988.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selected = comparison.groupby("task", group_keys=False).apply(choose_best).reset_index(drop=True)


,task,source,estimated_score,memory_bytes,params,filesize,ready,onnx
1,1,sample,17.047385,2819,24,1082,True,Sample/submission/task001.onnx
3,2,sample,13.514518,97200,93,14560,True,Sample/submission/task002.onnx
5,3,sample,17.918291,1172,18,5000,True,Sample/submission/task003.onnx
7,4,sample,14.084548,54000,1020,2522,True,Sample/submission/task004.onnx
9,5,sample,13.328150,116889,336,10910,True,Sample/submission/task005.onnx
11,6,sample,17.603051,1584,47,1124,True,Sample/submission/task006.onnx
13,7,sample,16.108351,7240,31,1258,True,Sample/submission/task007.onnx
15,8,sample,14.944307,23228,60,7647,True,Sample/submission/task008.onnx
17,9,sample,13.694825,77400,3841,30956,True,Sample/submission/task009.onnx
19,10,sample,16.479811,4939,76,6176,True,Sample/submission/task010.onnx


In [6]:
display_cols = [
    "task", "source", "selected", "estimated_score", "memory_bytes", "params", "filesize", "ready",
    "arc_agi_pass", "arc_agi_fail", "arc_gen_pass", "arc_gen_fail", "seconds", "onnx",
]
selected[display_cols].style.format({
    "estimated_score": "{:.3f}",
    "memory_bytes": "{:,}",
    "params": "{:,}",
    "filesize": "{:,}",
    "seconds": "{:.1f}",
})

,task,source,selected,estimated_score,memory_bytes,params,filesize,ready,arc_agi_pass,arc_agi_fail,arc_gen_pass,arc_gen_fail,seconds,onnx
0,1,outputs,False,15.272,"16,620",156,"3,374",True,6,0,262,0,0.6,outputs/task001.onnx
1,1,sample,True,17.047,"2,819",24,"1,082",True,6,0,262,0,0.4,Sample/submission/task001.onnx
2,2,outputs,False,11.553,"691,404",46,"13,751",True,6,0,262,0,6.4,outputs/task002.onnx
3,2,sample,True,13.515,"97,200",93,"14,560",True,6,0,262,0,3.0,Sample/submission/task002.onnx
4,3,outputs,False,13.771,"74,308",985,"10,277",True,4,0,261,0,2.6,outputs/task003.onnx
5,3,sample,True,17.918,"1,172",18,"5,000",True,4,0,261,0,1.4,Sample/submission/task003.onnx
6,4,outputs,False,12.113,"394,040","1,256","31,356",True,3,0,262,0,5.8,outputs/task004.onnx
7,4,sample,True,14.085,"54,000","1,020","2,522",True,3,0,262,0,0.4,Sample/submission/task004.onnx
8,5,outputs,False,10.466,"2,049,212","1,544","85,748",True,4,0,262,0,17.2,outputs/task005.onnx
9,5,sample,True,13.328,"116,889",336,"10,910",True,4,0,262,0,3.1,Sample/submission/task005.onnx


## Override Sample/submission

選ばれたモデルが `outputs/` 側の場合だけ、対応する `Sample/submission/taskXXX.onnx` をバックアップしてから上書きする。

In [7]:
def override_sample_with_selected(selected_summary):
    copied = []
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    for _, row in selected_summary.iterrows():
        task_num = int(row["task"])
        if row["source"] != "outputs":
            continue
        src = output_files[task_num]
        dst = sample_files[task_num]
        backup = BACKUP_DIR / dst.name
        shutil.copy2(dst, backup)
        shutil.copy2(src, dst)
        copied.append({
            "task": task_num,
            "src": str(src.relative_to(ROOT)),
            "dst": str(dst.relative_to(ROOT)),
            "backup": str(backup.relative_to(ROOT)),
        })
    return pd.DataFrame(copied)


override_log = override_sample_with_selected(selected_summary)
override_log

,task,src,dst,backup
0,255,outputs/task255.onnx,Sample/submission/task255.onnx,artifacts/sample-submission-backups/20260610_1...


## Create final submission.zip

`Sample/submission/` の400個の `taskXXX.onnx` を `outputs/submission.zip` にまとめる。

In [8]:
def create_submission_zip(source_dir=SAMPLE_SUBMISSION_DIR, zip_path=FINAL_ZIP):
    files = sorted(source_dir.glob("task*.onnx"))
    if len(files) != 400:
        raise ValueError(f"expected 400 ONNX files, found {len(files)}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in files:
            zf.write(path, arcname=path.name)
    return {
        "zip": str(zip_path.relative_to(ROOT)),
        "files": len(files),
        "bytes": zip_path.stat().st_size,
    }


zip_info = create_submission_zip()
zip_info

{'zip': 'outputs/submission.zip', 'files': 400, 'bytes': 711336}

In [9]:
print(f"selected outputs overrides: {len(override_log)}")
print(f"final zip: {zip_info['zip']} ({zip_info['bytes']:,} bytes)")
print(f"backup dir: {BACKUP_DIR.relative_to(ROOT)}")
print(f"failures: {len(failures)}")
failures

selected outputs overrides: 1
final zip: outputs/submission.zip (711,336 bytes)
backup dir: artifacts/sample-submission-backups/20260610_194445
failures: 0


{}